# 33 · Regression against statsmodels

**Question.** Fieldwork's Regression fits logistic and Poisson models in its own Rust kernel (Fieldwork experiment 167).
Does it give the coefficients, standard errors and intervals that statsmodels gives for the same declared model?

Two published models, the two the widget was built to reproduce:

- **Salmonella**, Germany 2021: cases against two control groups, Galia melon in April adjusted for age group and sex,
  likelihood-ratio intervals as published. The rows (CC BY 4.0) are in the fixture.
- **NYC**, 2016 to 2025: daily child asthma visits on ozone, adjusted for PM2.5, NO2, a temperature spline and holidays,
  840 strata, HAC errors. The counts are the city health department's and are **not** redistributed: the check fetches
  the authors' package from Zenodo and checks its digest.

statsmodels fits each model with its own IRLS and patsy's spline; the strata are fixed effects, a different route to the
conditional model's slopes. Likelihood-ratio limits are found here by refitting statsmodels' model with the coefficient
held fixed.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "numpy", "scipy", "statsmodels", "patsy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

## Parity

Criteria, fixed before running: coefficients and model-based standard errors within 1e-6 (relative), likelihood-ratio
limits within 1e-4, HAC standard errors within 1e-6 against statsmodels' Newey-West at 14 lags, and the ozone estimate
equal to the authors' archived R result within 1e-9.

**Amended after the first run, and recorded as such:** statsmodels is fitted to a tolerance of 1e-12. At its default it
stopped Salmonella one iteration early and its standard error for Galia melon differed from Fieldwork's by 4e-6;
tightened, the two agree to 15 digits. The NYC case fetches about 9 MB from Zenodo and can be slow.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_33_regression import run, rel

report = run()
print("agrees:", report["agrees"])
print()
for c in report["comparisons"]:
    compared = [t for t in c["terms"] if not t.get("separated")]
    worst = max(max(rel(*t["coefficient"]), rel(*t["se"])) for t in compared)
    print(f"  {'OK ' if c['agrees'] else 'DIFF'} {c['fixture']:<10} {c['errors']:<8} {c['rows']['fieldwork']:>5} rows  "
          f"worst relative difference {worst:.1e}")
    print(f"       {c['note']}")

## The estimates a reader checks

The exposure's ratio and interval, beside statsmodels' and the published figure.

In [ ]:
import math
for c in report["comparisons"]:
    first = c["terms"][0]
    b, b_sm = first["coefficient"]
    line = f"{c['fixture']:<10} {c['errors']:<8} {first['term']:<16} Fieldwork {math.exp(b):.4f}  statsmodels {math.exp(b_sm):.4f}"
    if "profile" in first:
        (lo, hi), (lo2, hi2) = first["profile"]["fieldwork"], first["profile"]["here"]
        line += f"   interval {lo:.2f}-{hi:.0f} against {lo2:.2f}-{hi2:.0f}"
    print(line)
    if "authorsArchivedR" in c:
        a = c["authorsArchivedR"]
        print(f"{'':>10} authors' R: coefficient {a['coefficient'][1]}  HAC SE {a['se'][1]}  agrees {a['agrees']}")

## A finding against Fieldwork: separation

Against the panel controls, nobody aged 80 or over is a case (or every one is), so that age group's odds ratio has **no
finite estimate**: the likelihood keeps rising as the coefficient grows. Both implementations stop somewhere along that
ridge, with standard errors in the tens of thousands or more. They agree that it is separated. **Fieldwork reports it as
an ordinary odds ratio with no warning**, which this check records and does not excuse.

In [ ]:
for c in report["comparisons"]:
    for t in c["terms"]:
        if t.get("separated"):
            r = t["fieldworkReported"]
            print(f"{c['note'][:44]}...: {t['term']}, standard errors {t['se'][0]:.0f} (Fieldwork) and {t['se'][1]:.0f} (statsmodels)")
            print(f"  Fieldwork reported {r['ratio']:.3g} ({r['lower']:.3g} to {r['upper']:.3g})")

## What this cannot establish

That either model is well specified, or that an adjusted estimate is causal. Offsets and overdispersed errors are not
exercised: neither example uses them.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
for f in report["fixtures"]:
    print("fixture    :", f["id"], f["sha256"][:16], "· Fieldwork", f["application"]["version"], "· release", f["operation"]["release"])
for k in ("statsmodels", "patsy", "pandas", "numpy"):
    print(f"{k:<11}:", report["environment"][k])
print("runtime    :", report["environment"]["runtime"])